In [ ]:
# Install the Python packages required by this notebook.
%pip install -q duckdb pyarrow scikit-learn xgboost scipy joblib

In [ ]:
# Mount Google Drive so this notebook can access the private MIMIC-IV data and derived files.
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Define the MIMIC-IV folders and the shared derived-data folder used by all notebooks.
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/Early Acute Kidney Injury Prediction + Production Monitoring/data")
HOSP_DIR = DATA_ROOT / "hosp"
ICU_DIR = DATA_ROOT / "icu"
DERIVED_DIR = DATA_ROOT / "derived"
DERIVED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Import the libraries used to extract and aggregate leakage-safe clinical features.
import duckdb
import numpy as np
import pandas as pd

In [ ]:
# Load the eligible cohort and the AKI labels generated by the previous notebooks.
eligible_cohort = pd.read_parquet(DERIVED_DIR / "eligible_cohort.parquet")
labels = pd.read_parquet(DERIVED_DIR / "aki_labels.parquet")

for frame in [eligible_cohort, labels]:
    for col in ["intime", "outtime", "prediction_time", "prediction_end", "admittime", "dischtime"]:
        if col in frame.columns:
            frame[col] = pd.to_datetime(frame[col])

model_cohort = eligible_cohort[
    eligible_cohort["stay_id"].isin(labels["stay_id"])
].copy()

print("Modeling ICU stays:", f"{len(model_cohort):,}")

In [ ]:
# Define the laboratory, vital-sign, urine-output, and weight item IDs used as model features.
LAB_ITEMS = {
    50912: "creatinine",
    51006: "bun",
    50983: "sodium",
    50971: "potassium",
    50882: "bicarbonate",
    50931: "glucose",
    50813: "lactate",
    51222: "hemoglobin",
    51265: "platelets",
    51301: "wbc",
}

VITAL_ITEMS = {
    220045: "heart_rate",
    220052: "map",
    220181: "map",
    220210: "resp_rate",
    220277: "spo2",
    223761: "temperature_f",
    223762: "temperature_c",
}

URINE_OUTPUT_ITEMIDS = [
    226559, 226560, 226561, 226584, 226563,
    226564, 226565, 226567, 226557, 226558,
    227488, 227489,
]

WEIGHT_ITEMIDS = [226512, 224639]

In [ ]:
# Extract selected laboratory measurements recorded between ICU admission and the 12-hour prediction time.
labevents_path = str(HOSP_DIR / "labevents.csv.gz")

con = duckdb.connect()
con.register(
    "cohort_df",
    model_cohort[["subject_id", "stay_id", "intime", "prediction_time"]]
)

lab_ids_sql = ", ".join(str(x) for x in LAB_ITEMS)

lab_events = con.execute(f"""
    SELECT
        c.stay_id,
        CAST(le.charttime AS TIMESTAMP) AS charttime,
        le.itemid,
        CAST(le.valuenum AS DOUBLE) AS value
    FROM cohort_df c
    INNER JOIN read_csv_auto('{labevents_path}') le
        ON c.subject_id = le.subject_id
       AND le.itemid IN ({lab_ids_sql})
       AND le.valuenum IS NOT NULL
       AND CAST(le.charttime AS TIMESTAMP) >= c.intime
       AND CAST(le.charttime AS TIMESTAMP) <= c.prediction_time
""").df()

con.close()

lab_events["feature"] = lab_events["itemid"].map(LAB_ITEMS)
print("Laboratory events:", f"{len(lab_events):,}")

In [ ]:
# Extract selected ICU vital signs recorded between ICU admission and the 12-hour prediction time.
chartevents_path = str(ICU_DIR / "chartevents.csv.gz")

con = duckdb.connect()
con.register(
    "cohort_df",
    model_cohort[["stay_id", "intime", "prediction_time"]]
)

vital_ids_sql = ", ".join(str(x) for x in VITAL_ITEMS)

vital_events = con.execute(f"""
    SELECT
        c.stay_id,
        CAST(ce.charttime AS TIMESTAMP) AS charttime,
        ce.itemid,
        CAST(ce.valuenum AS DOUBLE) AS value
    FROM cohort_df c
    INNER JOIN read_csv_auto('{chartevents_path}') ce
        ON c.stay_id = ce.stay_id
       AND ce.itemid IN ({vital_ids_sql})
       AND ce.valuenum IS NOT NULL
       AND CAST(ce.charttime AS TIMESTAMP) >= c.intime
       AND CAST(ce.charttime AS TIMESTAMP) <= c.prediction_time
""").df()

con.close()

vital_events["feature"] = vital_events["itemid"].map(VITAL_ITEMS)

fahrenheit_mask = vital_events["feature"].eq("temperature_f")
vital_events.loc[fahrenheit_mask, "value"] = (
    vital_events.loc[fahrenheit_mask, "value"] - 32.0
) * 5.0 / 9.0
vital_events.loc[fahrenheit_mask, "feature"] = "temperature_c"

print("Vital-sign events:", f"{len(vital_events):,}")

In [ ]:
# Remove clearly implausible laboratory and vital-sign values before feature aggregation.
PLAUSIBLE_RANGES = {
    "heart_rate": (20, 300),
    "map": (20, 220),
    "resp_rate": (1, 80),
    "spo2": (50, 100),
    "temperature_c": (25, 45),
    "creatinine": (0.1, 30),
    "bun": (1, 300),
    "sodium": (90, 200),
    "potassium": (1.5, 10),
    "bicarbonate": (2, 60),
    "glucose": (20, 1500),
    "lactate": (0.1, 40),
    "hemoglobin": (2, 25),
    "platelets": (1, 2000),
    "wbc": (0.1, 300),
}

def filter_plausible(events):
    events = events.dropna(subset=["feature", "value"]).copy()

    keep = pd.Series(True, index=events.index)

    for feature, (low, high) in PLAUSIBLE_RANGES.items():
        feature_mask = events["feature"].eq(feature)
        keep &= (~feature_mask) | events["value"].between(low, high)

    return events[keep].copy()

lab_events = filter_plausible(lab_events)
vital_events = filter_plausible(vital_events)

In [ ]:
# Aggregate each laboratory and vital-sign variable into minimum, maximum, mean, and last-value features.
def aggregate_events(events):
    events = events.sort_values(["stay_id", "feature", "charttime"]).copy()

    stats = (
        events
        .groupby(["stay_id", "feature"])["value"]
        .agg(["min", "max", "mean"])
        .reset_index()
    )

    last_values = (
        events
        .groupby(["stay_id", "feature"], as_index=False)
        .tail(1)[["stay_id", "feature", "value"]]
        .rename(columns={"value": "last"})
    )

    stats = stats.merge(last_values, on=["stay_id", "feature"], how="left")

    pieces = []

    for stat in ["min", "max", "mean", "last"]:
        wide = stats.pivot(index="stay_id", columns="feature", values=stat)
        wide.columns = [f"{column}_{stat}_12h" for column in wide.columns]
        pieces.append(wide)

    return pd.concat(pieces, axis=1).reset_index()

lab_features = aggregate_events(lab_events)
vital_features = aggregate_events(vital_events)

print("Laboratory feature columns:", lab_features.shape[1] - 1)
print("Vital-sign feature columns:", vital_features.shape[1] - 1)

In [ ]:
# Extract the first valid ICU weight available by the prediction time for each modeling stay.
con = duckdb.connect()
con.register(
    "cohort_df",
    model_cohort[["stay_id", "prediction_time"]]
)

weight_ids_sql = ", ".join(str(x) for x in WEIGHT_ITEMIDS)

weight_features = con.execute(f"""
    WITH ranked_weights AS (
        SELECT
            c.stay_id,
            CAST(ce.charttime AS TIMESTAMP) AS charttime,
            ce.itemid,
            CAST(ce.valuenum AS DOUBLE) AS weight_kg,
            ROW_NUMBER() OVER (
                PARTITION BY c.stay_id
                ORDER BY
                    CASE WHEN ce.itemid = 226512 THEN 0 ELSE 1 END,
                    CAST(ce.charttime AS TIMESTAMP)
            ) AS rn
        FROM cohort_df c
        INNER JOIN read_csv_auto('{chartevents_path}') ce
            ON c.stay_id = ce.stay_id
           AND ce.itemid IN ({weight_ids_sql})
           AND ce.valuenum IS NOT NULL
           AND CAST(ce.valuenum AS DOUBLE) BETWEEN 20 AND 400
           AND CAST(ce.charttime AS TIMESTAMP) <= c.prediction_time
    )
    SELECT stay_id, weight_kg
    FROM ranked_weights
    WHERE rn = 1
""").df()

con.close()

In [ ]:
# Calculate total 12-hour urine output and normalized urine-output rate for each modeling stay.
outputevents_path = str(ICU_DIR / "outputevents.csv.gz")

con = duckdb.connect()
con.register(
    "cohort_df",
    model_cohort[["stay_id", "intime", "prediction_time"]]
)

urine_ids_sql = ", ".join(str(x) for x in URINE_OUTPUT_ITEMIDS)

urine_features = con.execute(f"""
    SELECT
        c.stay_id,
        SUM(
            CASE
                WHEN oe.itemid = 227488 AND CAST(oe.value AS DOUBLE) > 0
                    THEN -1.0 * CAST(oe.value AS DOUBLE)
                ELSE CAST(oe.value AS DOUBLE)
            END
        ) AS urine_output_total_12h
    FROM cohort_df c
    INNER JOIN read_csv_auto('{outputevents_path}') oe
        ON c.stay_id = oe.stay_id
       AND oe.itemid IN ({urine_ids_sql})
       AND oe.value IS NOT NULL
       AND CAST(oe.charttime AS TIMESTAMP) >= c.intime
       AND CAST(oe.charttime AS TIMESTAMP) <= c.prediction_time
    GROUP BY c.stay_id
""").df()

con.close()

urine_features = urine_features.merge(weight_features, on="stay_id", how="left")
urine_features["urine_output_ml_kg_h_12h"] = (
    urine_features["urine_output_total_12h"]
    / urine_features["weight_kg"]
    / 12.0
)

In [ ]:
# Assemble demographics, clinical features, and the AKI target into one row per ICU stay.
base_features = model_cohort[
    [
        "subject_id",
        "hadm_id",
        "stay_id",
        "age",
        "gender",
        "race",
        "first_careunit",
        "anchor_year_group",
    ]
].copy()

base_features = base_features.merge(
    labels[["stay_id", "target"]],
    on="stay_id",
    how="inner",
    validate="one_to_one",
)

ml_dataset = base_features.merge(lab_features, on="stay_id", how="left")
ml_dataset = ml_dataset.merge(vital_features, on="stay_id", how="left")
ml_dataset = ml_dataset.merge(urine_features, on="stay_id", how="left")

print("ML dataset shape:", ml_dataset.shape)
print("Positive rate:", f"{ml_dataset['target'].mean():.3%}")
ml_dataset.head()

In [ ]:
# Inspect feature missingness to understand which variables are sparsely measured in the first 12 ICU hours.
missingness = (
    ml_dataset
    .isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_rate")
    .to_frame()
)

missingness.head(25)

In [ ]:
# Save the final one-row-per-ICU-stay feature table for model training.
ml_dataset.to_parquet(DERIVED_DIR / "ml_dataset.parquet", index=False)
print("Saved:", DERIVED_DIR / "ml_dataset.parquet")